In [1]:
import pandas as pd
import numpy as np

In [2]:
sites = pd.read_csv('Data/Solar_Site_Details.csv')
energy = pd.read_csv('Data/Solar_Energy_Generation.csv')
weather = pd.read_csv('Data/Weather_Data_reordered_all.csv')

In [3]:
energy.shape

(2731946, 4)

In [4]:
sites.loc[sites['kWp'].notna(), 'CampusKey'].unique()

array([1])

In [5]:
sites.head()

,CampusKey,SiteKey,kWp,Number of panels,Panel,Inverter,Optimizers,Metric,lat,Lon
0,2,1,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
1,2,2,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
2,2,3,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
3,2,4,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679
4,2,5,NaN,NaN,NaN,NaN,NaN,NaN,-36.111209,146.848679


In [6]:
energy = energy[energy['CampusKey'] == 1]
weather = weather[weather['CampusKey']==1]
sites = sites[sites['CampusKey']==1]

In [7]:
print(energy['CampusKey'].unique())
print(weather['CampusKey'].unique())
print(sites['CampusKey'].unique())

[1]
[1]
[1]


In [8]:
sites[sites['kWp'].isna()][['SiteKey']]

,SiteKey
27,28
28,29


In [9]:
sites = sites.dropna(subset=['kWp'])

In [10]:
sites['SiteKey'].values

array([14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 30, 31, 32,
       33, 34, 35, 36, 37, 38, 39, 40])

In [11]:
energy = energy.drop(columns=['CampusKey'])
weather = weather.drop(columns=['CampusKey'])
sites = sites.drop(columns=['CampusKey'])

In [12]:
energy.head()

,SiteKey,Timestamp,SolarGeneration
853089,14,2020-06-14 00:15:00,NaN
853090,14,2020-06-14 00:30:00,NaN
853091,14,2020-06-14 00:45:00,NaN
853092,14,2020-06-14 01:00:00,NaN
853093,14,2020-06-14 01:15:00,NaN


In [13]:
weather.head()

,Timestamp,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection
0,1/1/2020 0:00,13.666667,13.880000,8.960000,72.400000,0.000000,188.133333
1,1/1/2020 0:15,13.206667,13.666667,9.040000,73.466667,1.200000,203.866667
2,1/1/2020 0:30,12.840000,13.553333,9.053333,74.000000,2.520000,222.800000
3,1/1/2020 0:45,12.113333,13.506667,9.100000,74.466667,5.986667,231.133333
4,1/1/2020 1:00,11.946667,13.260000,9.266667,76.533333,5.946667,247.866667


In [14]:
energy_only = set(energy['SiteKey']) - set(sites['SiteKey'])
sites_only = set(sites['SiteKey']) - set(energy['SiteKey'])

print("Only in energy:", energy_only)
print("Only in sites:", sites_only)

Only in energy: {28, 29}
Only in sites: set()


In [15]:
energy = energy[~energy['SiteKey'].isin(energy_only)]

In [16]:
energy.shape

(1599868, 3)

In [17]:
print(energy["Timestamp"].iloc[0])
print(weather["Timestamp"].iloc[0])

2020-06-14 00:15:00
1/1/2020 0:00


In [18]:
energy["Timestamp"] = pd.to_datetime(energy["Timestamp"])
weather["Timestamp"] = pd.to_datetime(weather["Timestamp"])

data = pd.merge(energy, weather, on=['Timestamp'], how='left')
data = pd.merge(data, sites, on='SiteKey', how='left')

In [19]:
data.head()

,SiteKey,Timestamp,SolarGeneration,ApparentTemperature,AirTemperature,DewPointTemperature,RelativeHumidity,WindSpeed,WindDirection,kWp,Number of panels,Panel,Inverter,Optimizers,Metric,lat,Lon
0,14,2020-06-14 00:15:00,NaN,12.140000,12.853333,12.626667,98.733333,7.880000,21.133333,66.0,200.0,Trina 330W,2 x SolarEdge SE27.6K,100 x P730,kWh,-37.718287,145.050975
1,14,2020-06-14 00:30:00,NaN,12.740000,12.700000,12.500000,99.000000,3.720000,55.400000,66.0,200.0,Trina 330W,2 x SolarEdge SE27.6K,100 x P730,kWh,-37.718287,145.050975
2,14,2020-06-14 00:45:00,NaN,12.253333,12.593333,12.393333,99.000000,5.706667,63.933333,66.0,200.0,Trina 330W,2 x SolarEdge SE27.6K,100 x P730,kWh,-37.718287,145.050975
3,14,2020-06-14 01:00:00,NaN,11.593333,12.613333,12.413333,99.000000,9.133333,151.600000,66.0,200.0,Trina 330W,2 x SolarEdge SE27.6K,100 x P730,kWh,-37.718287,145.050975
4,14,2020-06-14 01:15:00,NaN,10.973333,12.820000,12.046667,95.133333,12.760000,341.266667,66.0,200.0,Trina 330W,2 x SolarEdge SE27.6K,100 x P730,kWh,-37.718287,145.050975


In [20]:
data.shape

(1599868, 17)

In [21]:
data.to_csv('Data/Joined_Data.csv',index=False)